# 49. A precision fix in the strict threshold, the numbers it touches, and why exact numbering fails

CPU only, a few minutes. Two things.

1. **The strict threshold in the scores' own precision.** When the top benign scores tie in a group larger than the cap, `strict_threshold` (notebook 47) moves the threshold just above the largest benign score. It computed that next value in 64-bit precision. Most score caches are 32-bit, and numpy compares a 32-bit array with a Python float in 32-bit precision, so the value rounded back to the largest benign score and blocked the tied group again. Every miss rate, blocked count and leave-one-out rate in notebooks 47 and 48 converted the scores to 64-bit first and is correct. Three things compared the 32-bit arrays directly and are affected: the AgentDojo bootstrap intervals for detectors where the fallback fired (Table 16), the prose-control intervals for ProtectAI-v1, and the NotInject false-positive rate in one row of Table 7 (ProtectAI-v2 calibrated on NotInject). The fix returns the next representable value in the scores' own precision; the three outputs are recomputed and compared with the earlier files.
2. **Why exact numbering fails.** Notebook 48 found that the final Qwen 7B model completes the numbering task exactly on only 0.475 of items, for statements and instructions alike, against 0.925 for the undefended 7B on statements, and the final 3B on 0.688 of instruction items against 0.900 of statement items. Every failure is sorted into a reason (wrong number of lines, an unnumbered line, numbers out of order, changed words, a preamble without a colon), with examples, for every model, so the paper can say what the measure is catching.

**Outputs.** `src/strict_threshold.py` (fixed), `reports/agentdojo_eval_strict.csv`, `reports/agentdojo_by_attack_strict.csv`, `reports/agentdojo_by_suite_strict.csv`, `reports/prose_control_eval.csv`, `reports/calibration_source_docfpr_v2.csv` (recomputed), `reports/precision_fix_changes.csv`, `reports/fidelity_failure_reasons.csv`.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
!pip install -q scikit-learn
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports', 'figures'): os.makedirs(d_, exist_ok=True)
print('Session ready (CPU is enough)')

## Stage A. The fixed routine and its self-check

In [ ]:
%%writefile src/strict_threshold.py
"""A guaranteed threshold that always satisfies the false-positive cap.

Decision rule everywhere in this project: block a text if its score is at or above the threshold. For a target
false-positive rate f on a calibration set of benign scores, the guaranteed threshold is the smallest s such that the
share of calibration benigns at or above s is at most f. score_diagnostics.thresholds searches s among the benign
scores themselves and, when none qualifies (the top scores tie and the tied group is larger than f allows), falls
back to the largest benign score, which blocks the whole tied group and exceeds the cap. strict_threshold instead
takes s just above the largest benign score in that case: it blocks no calibration benign, so the cap holds, and the
miss rate at s is the share of injections scoring at or below the largest benign. Where a qualifying benign score
exists the two routines agree exactly.

Precision. "Just above" is the next representable value in the scores' own floating-point type. numpy compares a
32-bit array with a Python float in 32-bit precision, so a value computed in 64-bit precision would round back to the
largest benign score and block the tied group again (notebook 49)."""
from __future__ import annotations
import numpy as np


def strict_threshold(p_ben, fpr=0.01):
    """Returns (threshold, blocked count on the calibration set, achieved calibration FPR)."""
    a = np.asarray(p_ben); dt = a.dtype if a.dtype.kind == "f" else np.dtype(np.float64)
    p = a.astype(np.float64); cand = np.unique(p)
    for s in cand:
        if (p >= s).mean() <= fpr: return float(s), int((p >= s).sum()), float((p >= s).mean())
    s = float(np.nextafter(dt.type(cand[-1]), dt.type(np.inf)))
    assert not (a >= dt.type(s)).any() and not (p >= s).any()
    return s, 0, 0.0


def loo_fpr(p_ben, fpr=0.01):
    """Leave-one-out estimate of the deployment false-positive rate of strict_threshold: for each benign, the threshold
    is set on the others and the held-out benign is scored against it."""
    a = np.asarray(p_ben); n = len(a); hits = 0
    for i in range(n):
        s, _, _ = strict_threshold(np.delete(a, i), fpr); hits += int(float(a[i]) >= s)
    return hits / n


def fnr(p_atk, t):
    p_atk = np.asarray(p_atk, float); return float((p_atk < t).mean()) if len(p_atk) else float("nan")


In [ ]:
import importlib, json, hashlib, collections, numpy as np, pandas as pd
import strict_threshold, tie_interval, interval_panel, prose_control, fidelity_audit, fidelity_exact, ci_tools
[importlib.reload(m) for m in (strict_threshold, tie_interval, interval_panel, prose_control, fidelity_audit, fidelity_exact, ci_tools)]
from strict_threshold import strict_threshold as strict, loo_fpr, fnr
from tie_interval import convention_interval, fnr_low, fnr_high, fpr_low_end, fpr_high_end, guaranteed, top_tie, fnr_at
from interval_panel import three_thresholds
from sklearn.metrics import roc_auc_score
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 300)
# a 32-bit tied top group: the threshold must block none of it whether the comparison is done in 32 or 64 bits
pb = np.r_[np.random.default_rng(0).uniform(0, 0.5, 123), np.full(6, 0.99999)].astype(np.float32)
t, k, f = strict(pb); assert k == 0 and not (pb >= t).any() and not (pb.astype(float) >= t).any(), 'precision fix failed'
pb2 = np.random.default_rng(1).uniform(0, 1, 399).astype(np.float32); assert strict(pb2)[0] == three_thresholds(pb2)['guar'], 'the routines must agree when a qualifying benign score exists'
assert guaranteed(pb)[0] == t, 'tie_interval must use the fixed routine'
print('self-check passed: a tied 32-bit top group of 6 of 129 is blocked by', int((pb >= t).sum()), 'in 32-bit and', int((pb.astype(float) >= t).sum()), 'in 64-bit comparisons')
CHANGES = []
def note(where, item, before, after): CHANGES.append(dict(output=where, item=item, before=before, after=after, changed=bool(str(before) != str(after))))

## Stage B. Recompute the three affected outputs

In [ ]:
# B1. AgentDojo (Tables 16 and 17), as notebook 47 Stage C, with the fixed routine
from agentdojo_source import ATTACKS, SUITES
R_ALL = json.load(open('data/agentdojo/set.json')); WINDOW_CHARS = 1600
KEEP = np.array([i for i, r in enumerate(R_ALL) if r['label'] == 0 or r.get('inj_offset', 0) <= WINDOW_CHARS]); R = [R_ALL[i] for i in KEEP]
Y = np.array([r['label'] for r in R]); SUITE = np.array([r['suite'] for r in R]); ATK = np.array([r['attack'] for r in R]); SLOT = np.array([f"{r['suite']}|{r['slot']}" for r in R]); ben = Y == 0; atk = Y == 1
LABEL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)', 'deepset_injection': 'deepset injection', 'protectai_v1': 'ProtectAI-v1', 'fmops_distilbert': 'fmops distilbert',
         'full': 'structural (full)', 'full_aug': 'structural (full_aug)', 'conventional_ft': 'conventional fine-tune', 'piguard': 'PIGuard'}
SA = {nm: np.load(f'data/agentdojo/score_{t}.npy')[KEEP] for t, nm in LABEL.items() if os.path.exists(f'data/agentdojo/score_{t}.npy')}
from structural_train import load_jsonl
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
STR_TRAIN = load_jsonl('data/structural/train.jsonl'); vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200000, sublinear_tf=True)
lr = LogisticRegression(C=1.0, max_iter=2000).fit(vec.fit_transform([r['text'] for r in STR_TRAIN]), [r['label'] for r in STR_TRAIN]); SA['TF-IDF reference'] = lr.predict_proba(vec.transform([r['text'] for r in R]))[:, 1]
print('score precision:', {k: str(v.dtype) for k, v in SA.items()})
OLD = pd.read_csv('reports/agentdojo_eval.csv').set_index('model'); PREV = pd.read_csv('reports/agentdojo_eval_strict.csv').set_index('model')
def cluster_boot(fn, B=1000, seed=0):
    rng = np.random.default_rng(seed); slots = np.unique(SLOT[atk]); bidx = np.where(ben)[0]; vals = []
    for _ in range(B):
        sl = rng.choice(slots, len(slots), replace=True); ai = np.concatenate([np.where((SLOT == s) & atk)[0] for s in sl]); bi = rng.choice(bidx, len(bidx), replace=True)
        try: vals.append(fn(ai, bi))
        except Exception: pass
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
rows, arows, srows = [], [], []
for name, p in SA.items():
    told = three_thresholds(p[ben])['guar']; ts, k, f_ = strict(p[ben]); lo, hi = cluster_boot(lambda ai, bi: fnr(p[ai], ts))
    rows.append(dict(model=name, AUROC=round(float(OLD.AUROC.get(name, roc_auc_score(Y, p))), 3), t_strict=ts, blocked_strict=k, calFPR_strict=round(f_, 4), loo_FPR_strict=round(loo_fpr(p[ben]), 4),
                     FNR_strict=round(fnr(p[atk], ts), 3), FNR_strict_lo95=round(lo, 3), FNR_strict_hi95=round(hi, 3), FNR_old=round(fnr(p[atk], told), 3), calFPR_old=round(float((p[ben] >= told).mean()), 4), fallback_fired=bool(ts != told)))
    for a in ATTACKS:
        m = atk & (ATK == a)
        if m.sum(): arows.append(dict(model=name, attack=a, n=int(m.sum()), FNR_strict=round(fnr(p[m], ts), 3), FNR_old=round(fnr(p[m], told), 3)))
    for s in SUITES:
        ma = atk & (SUITE == s); mb = ben & (SUITE == s)
        if ma.sum() and mb.sum() >= 10: srows.append(dict(model=name, suite=s, n_atk=int(ma.sum()), n_ben=int(mb.sum()), FNR_strict=round(fnr(p[ma], ts), 3)))
EVS = pd.DataFrame(rows); EVS.to_csv('reports/agentdojo_eval_strict.csv', index=False)
pd.DataFrame(arows).to_csv('reports/agentdojo_by_attack_strict.csv', index=False); pd.DataFrame(srows).to_csv('reports/agentdojo_by_suite_strict.csv', index=False)
for r in EVS.itertuples():
    for col in ('FNR_strict', 'FNR_strict_lo95', 'FNR_strict_hi95', 'blocked_strict', 'loo_FPR_strict'): note('Table 16 (AgentDojo)', f'{r.model}: {col}', PREV.loc[r.model, col], getattr(r, col))
print('=== AgentDojo, strict threshold, recomputed ==='); print(EVS.drop(columns=['t_strict']).to_string(index=False))
# B2. the prose control (Table 17, lower rows): intervals recomputed with the fixed routine
from prose_control import cluster_boot as pc_boot
PCS = json.load(open('data/prose_control/set.json')); SET_H = PCS['hash']; PC = PCS['records']
PC_Y = np.array([r['label'] for r in PC]); PC_SID = np.array([r['string_id'] for r in PC]); b_ = PC_Y == 0; a_ = PC_Y == 1
GROUPS = {s: np.where(a_ & (PC_SID == s))[0] for s in np.unique(PC_SID[a_])}; BIDX = np.where(b_)[0]
PCE_OLD = pd.read_csv('reports/prose_control_eval.csv'); rows = []
TAGOF = {'ProtectAI-v2': 'protectai_v2', 'Prompt-Guard-2 (86M)': 'prompt_guard_2', 'Prompt-Guard-2 (22M)': 'prompt_guard_2_22m', 'deepset injection': 'deepset_injection', 'ProtectAI-v1': 'protectai_v1',
         'fmops distilbert': 'fmops_distilbert', 'structural (full)': 'full', 'structural (full_aug)': 'full_aug', 'conventional fine-tune': 'conventional_ft', 'PIGuard': 'piguard'}
for o in PCE_OLD.itertuples():
    p = np.load(f'data/prose_control/score_{TAGOF[o.model]}_{SET_H}.npy'); ts, k, f_ = strict(p[b_])
    fl, fh = pc_boot(lambda ai, bi: fnr(p[ai], ts), GROUPS, BIDX); r = o._asdict(); r.pop('Index')
    r.update(blocked=k, calFPR=f_, loo_FPR=loo_fpr(p[b_]), FNR=fnr(p[a_], ts), FNR_lo95=fl, FNR_hi95=fh); rows.append(r)
    for col in ('FNR', 'FNR_lo95', 'FNR_hi95', 'blocked'): note('Table 17 (prose control)', f'{o.model}: {col}', round(float(getattr(o, col)), 3), round(float(r[col]), 3))
PCE = pd.DataFrame(rows); PCE.to_csv('reports/prose_control_eval.csv', index=False)
print('\n=== prose control, recomputed ==='); print(PCE.round(3).to_string(index=False))

In [ ]:
# B3. Table 7, the false-positive rates at each pool's guaranteed threshold, recomputed with the fixed routine
import data_loaders; importlib.reload(data_loaders)
from data_loaders import load_bipia_categorized, load_deepset, load_jailbreak, BIPIA_HIJACK, BIPIA_MALICIOUS
bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS); HOSTM = bip.label.values == 0; HIJ = bip.meta.isin(BIPIA_HIJACK).values
yd = load_deepset().label.values; yj = load_jailbreak().label.values
TAGS = ['protectai_v2', 'prompt_guard_2', 'prompt_guard_2_22m']; LAB3 = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
POOLS = {'deepset': ['deepset'], 'jailbreak_benign': ['jailbreak_benign'], 'notinject': ['notinject'], 'alpaca': ['alpaca'], 'dolly': ['dolly'], 'pooled (all four direct sources)': ['deepset', 'jailbreak_benign', 'alpaca', 'dolly']}
T7_OLD = pd.read_csv('reports/calibration_source_docfpr_v2.csv'); rows = []
for tag in TAGS:
    dsc, bsc, jsc, nsc = (np.load(f'data/score_{tag}_{k}.npy') for k in ('deepset', 'bipia_cat', 'jailbreak', 'notinject'))
    BEN = {'deepset': dsc[yd == 0], 'jailbreak_benign': jsc[yj == 0], 'notinject': nsc, 'alpaca': np.load(f'data/score_{tag}_alpaca.npy'), 'dolly': np.load(f'data/score_{tag}_dolly.npy')}
    hosts, hij = bsc[HOSTM], bsc[HIJ]
    for pname, parts in POOLS.items():
        pool = np.concatenate([BEN[s] for s in parts]); iv = convention_interval(pool); tg, kg, fg = guaranteed(pool); tt, cap = top_tie(pool)
        o = T7_OLD[(T7_OLD.detector == LAB3[tag]) & (T7_OLD.pool == pname)].iloc[0]
        r = dict(detector=LAB3[tag], pool=pname, N=len(pool), blocked_convention=iv['k'], blocked_guar=kg, calFPR_guar=fg, top_tie=tt, cap=cap, fallback_earlier=bool(tt > cap),
                 FNR_low_hijack=fnr_low(hij, iv), FNR_high_hijack=fnr_high(hij, iv), FNR_guar_hijack=fnr_at(hij, tg),
                 docFPR_low=fpr_low_end(hosts, iv), docFPR_high=fpr_high_end(hosts, iv), docFPR_guar=fpr_high_end(hosts, dict(v_hi=tg)),
                 notinjectFPR_low=fpr_low_end(BEN['notinject'], iv), notinjectFPR_high=fpr_high_end(BEN['notinject'], iv), notinjectFPR_guar=fpr_high_end(BEN['notinject'], dict(v_hi=tg)))
        rows.append(r)
        for col in ('FNR_guar_hijack', 'docFPR_guar', 'notinjectFPR_guar', 'blocked_guar'): note('Table 7 (calibration pools)', f'{LAB3[tag]}, {pname}: {col}', round(float(o[col]), 3), round(float(r[col]), 3))
T7 = pd.DataFrame(rows); T7.to_csv('reports/calibration_source_docfpr_v2.csv', index=False)
print('=== Table 7 recomputed ==='); print(T7.round(3).to_string(index=False))
CH = pd.DataFrame(CHANGES); CH.to_csv('reports/precision_fix_changes.csv', index=False)
print('\n=== every value that changed ==='); print(CH[CH.changed].to_string(index=False) if CH.changed.any() else 'none')
print(f'\n{int(CH.changed.sum())} of {len(CH)} checked values changed')

## Stage C. Why complete transformations fail

In [ ]:
from fidelity_exact import norm, split_sentences, score as fscore
from fidelity_audit import doc_sentences, complete, _lines, _units, _NUM
H = lambda s: hashlib.sha1(s.encode()).hexdigest()
FITEMS = json.load(open('data/fidelity_exact/items.json'))
FMODELS = [('undefended_3b', 'Qwen 3B undefended', 'data/fidelity_exact/out_undefended_3b.json'), ('twin_distilled_v2', 'Qwen 3B twin distillation v2', 'data/fidelity_exact/out_twin_distilled_v2.json'),
           ('fidelity_v4', 'Qwen 3B full recipe (final)', 'data/fidelity_exact/out_fidelity_v4.json'), ('undefended_7b', 'Qwen 7B undefended', 'data/fidelity_exact/out_undefended_7b.json'),
           ('twin_distilled_7b', 'Qwen 7B twin distillation', 'data/fidelity_exact/out_twin_distilled_7b.json'), ('fidelity_v4_7b', 'Qwen 7B literal phase', 'data/fidelity_exact/out_fidelity_v4_7b.json'),
           ('fidelity_v5_7b', 'Qwen 7B full recipe (final)', 'data/fidelity_exact/out_fidelity_v5_7b.json'), ('llama_undefended', 'Llama 8B undefended', 'data/llama/exec_llama_undefended.json'),
           ('llama_secalign', 'Meta-SecAlign-8B (as specified)', 'data/llama/exec_llama_secalign.json'), ('llama_full', 'Llama 8B ours, full recipe', 'data/llama/exec_llama_full.json')]
def reason(it, out, w):
    sents = doc_sentences(it, w); lines = _lines(out); task = it['task']
    if complete(it, out, w): return 'complete'
    if not (out or '').strip(): return 'empty'
    if task == 'number':
        ms = [_NUM.match(l) for l in lines]
        if len(lines) == len(sents) + 1 and not ms[0] and all(ms[1:]): return 'an unnumbered first line (preamble without a colon)'
        if len(lines) == len(sents) + 1 and not ms[-1] and all(ms[:-1]): return 'an unnumbered last line'
        if len(lines) > len(sents) and all(ms): return 'more numbered lines than sentences (a sentence split)'
        if len(lines) < len(sents) and all(ms): return 'fewer numbered lines than sentences (sentences merged or dropped)'
        if len(lines) == len(sents) and all(ms):
            if any(int(m.group(1)) != k + 1 for k, m in enumerate(ms)): return 'numbers out of order'
            return 'words changed in a numbered line'
        if len(lines) == 1: return 'not split into lines'
        return 'other line structure'
    u = _units(lines); want = [norm(x) for x in sents if norm(x)]
    if task == 'quote_last':
        if len(u) > 1 and norm(it[w]) in u: return 'the sentence plus other text'
        if norm(it[w]) not in ' '.join(u): return 'a different sentence or no quote'
        return 'words changed'
    missing = [x for x in want if x not in u]; extra = [x for x in u if x not in want]
    if missing and not extra: return 'a sentence missing'
    if extra and not missing: return 'text added'
    if missing and extra: return 'words changed or text replaced'
    return 'order changed'
rows, EXAMPLES = [], {}
for t, lab, f in FMODELS:
    if not os.path.exists(f): print('missing', f); continue
    O = json.load(open(f))
    for i, it in enumerate(FITEMS):
        for w in ('instruction', 'statement'):
            o = O[H(it[f'prompt_{w}'])]; rs = reason(it, o, w); rows.append(dict(model=lab, task=it['task'], version=w, reason=rs))
            if rs != 'complete': EXAMPLES.setdefault((lab, it['task'], w, rs), []).append((it, o))
RS = pd.DataFrame(rows)
TAB = RS.groupby(['model', 'task', 'version', 'reason']).size().rename('items').reset_index(); TAB.to_csv('reports/fidelity_failure_reasons.csv', index=False)
for task in ('number', 'repeat', 'quote_last'):
    print(f'\n=== {task}: reasons, items out of 80 ===')
    print(TAB[TAB.task == task].pivot_table(index=['model', 'version'], columns='reason', values='items', fill_value=0).to_string())
flat = lambda s, k=600: ' | '.join(l.strip() for l in str(s or '').splitlines() if l.strip())[:k]
for (lab, task, w) in (('Qwen 7B full recipe (final)', 'number', 'statement'), ('Qwen 7B full recipe (final)', 'number', 'instruction'), ('Qwen 3B full recipe (final)', 'number', 'instruction'),
                       ('Qwen 3B full recipe (final)', 'number', 'statement'), ('Qwen 7B undefended', 'number', 'statement')):
    keys = [k for k in EXAMPLES if k[0] == lab and k[1] == task and k[2] == w]
    print(f'\n##### {lab}, {task}, {w} version: failing examples by reason')
    for k in keys:
        print(f'--- {k[3]} ({len(EXAMPLES[k])} items)')
        for it, o in EXAMPLES[k][:2]:
            print('  EXPECTED |', flat('\n'.join(f'{j + 1}. {s}' for j, s in enumerate(doc_sentences(it, w)))))
            print('  OUTPUT   |', flat(o)); print()

## Stage D. Log and commit

In [ ]:
from reslog import log_result
summary = ('Precision fix, changed values:\n' + (CH[CH.changed].to_string(index=False) if CH.changed.any() else 'none') + '\n\nAgentDojo strict:\n' + EVS.drop(columns=['t_strict']).to_string(index=False)
           + '\n\nProse control:\n' + PCE.round(3).to_string(index=False) + '\n\nFidelity failure reasons (number):\n' + TAB[TAB.task == 'number'].pivot_table(index=['model', 'version'], columns='reason', values='items', fill_value=0).to_string())
log_result('nb49: strict threshold computed in the scores own precision; AgentDojo, prose control and Table 7 recomputed; reasons for incomplete transformations', summary, csv_df=CH, csv_name='precision_fix_changes.csv')
print('logged')

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py", "nb49: strict threshold takes the next value above a tied top group in the scores' own precision (a 64-bit value rounded back in 32-bit comparisons); AgentDojo strict table, prose-control intervals and Table 7 recomputed with every changed value listed; reasons for incomplete transformations on the fidelity items"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)